# Lab 2 — Parallelize a Catalog Enrichment Pipeline (AI-assisted)

**Time:** ~15–20 min  
**Mode:** Drive an AI coding assistant (Claude Code, Cursor, Copilot Chat, etc.)
to do most of the typing. You are the architect; the AI is the typist.

## Learning objectives
1. Practice prompting an AI assistant for a concrete Ray Core refactor.
2. Read AI-generated Ray code critically and verify correctness.
3. Recognize when to push back on the assistant (wrong API, missing actor handle, etc.).

## The starting point
Below is a **sequential** mini-enrichment pipeline that processes a list of
products. Your job is to convert it into a parallel Ray Core program — but you'll do it
by *prompting an AI assistant* rather than writing the code yourself.

In [ ]:
import ray, time, random

if not ray.is_initialized():
    ray.init()

PRODUCTS = [
    {"id": i, "cat": random.choice(["Housewares", "Apparel", "Snacks", "Books"]),
     "raw_desc": f"product number {i}"}
    for i in range(40)
]

def fetch_metadata(p):       # simulates DB lookup
    time.sleep(0.05)
    return {**p, "price": round(random.uniform(2, 200), 2)}

def score(p):                # simulates a model call
    time.sleep(0.20)
    return {**p, "score": random.random()}

def write(p):                # simulates a sink
    time.sleep(0.05)
    return p

%time results = [write(score(fetch_metadata(p))) for p in PRODUCTS]
print("processed:", len(results), "sample:", results[0])

## Exercise — drive the AI assistant

Open Claude Code (or your editor's AI chat) on this notebook and use the
prompts below. Run the produced code in the cells that follow, and verify each
acceptance criterion before moving on.

### Prompt A — convert to Ray tasks

> *"In this notebook, refactor `fetch_metadata`, `score`, and `write` into Ray
> remote tasks. Then process the `PRODUCTS` list end-to-end in parallel by
> chaining ObjectRefs (do not call `ray.get` between stages). Time the result
> with `%time`. Keep the output schema identical to the sequential version."*

**Acceptance criteria for A:**
- 40 results returned, each with keys `id, cat, raw_desc, price, score`.
- Wall time < ⅓ of the sequential version.
- No `ray.get` between stages — only at the very end.

### Prompt B — add a stats actor

> *"Add a Ray actor named `CategoryStats` that tracks (a) how many products it
> has seen per category, and (b) the highest score seen per category. Have the
> `score` task call `stats.record.remote(cat, score)`. Expose a method
> `summary()` that returns both views. Show the summary at the end."*

**Acceptance criteria for B:**
- Sum of per-category counts equals 40.
- For each category in `summary()['top_score']`, the value lies in [0, 1].
- Counts and top scores match what you'd compute from `results` directly.

### Prompt C — review and tighten (don't skip this!)

> *"Review the code you produced. Identify (1) any task that doesn't actually
> need to be remote, (2) any place where we are unintentionally serializing on
> `ray.get`, and (3) any race condition or correctness issue with the actor.
> Fix what you find and explain your reasoning."*

Read the answer carefully. **Disagree where appropriate.** AI assistants will
often invent issues that aren't there, or miss real ones. Common things to
verify yourself:
- Is the actor handle being passed into tasks (and not re-created per call)?
- Are we mutating the same dict from multiple tasks?
- Is `write` doing something that would actually need to be remote in production?

In [ ]:
# Paste / iterate on the AI-generated code here.
# Run it. Confirm the acceptance criteria from each prompt before moving on.


## Reflection
Briefly answer in a markdown cell or aloud with a partner:

1. What was the *first* thing the AI got wrong (if anything)? How did you catch it?
2. Was there anywhere you would have written it differently?
3. If the dataset were 10 million rows instead of 40, would Ray Core still be the right tool, or would you reach for Ray Data? Why?

---

# Reference solution

*One reasonable target — your AI's output may differ in style.*

In [ ]:
@ray.remote
def fetch_metadata_r(p):
    time.sleep(0.05)
    return {**p, "price": round(random.uniform(2, 200), 2)}

@ray.remote
def score_r(p, stats_handle):
    time.sleep(0.20)
    p = {**p, "score": random.random()}
    stats_handle.record.remote(p["cat"], p["score"])
    return p

@ray.remote
def write_r(p):
    time.sleep(0.05)
    return p

@ray.remote
class CategoryStats:
    def __init__(self):
        self.counts, self.top = {}, {}
    def record(self, cat, score):
        self.counts[cat] = self.counts.get(cat, 0) + 1
        self.top[cat] = max(self.top.get(cat, 0.0), score)
    def summary(self):
        return {"counts": dict(self.counts), "top_score": dict(self.top)}

stats = CategoryStats.remote()
%time results = ray.get([write_r.remote(score_r.remote(fetch_metadata_r.remote(p), stats)) for p in PRODUCTS])
print("processed:", len(results))
print("summary:", ray.get(stats.summary.remote()))